In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from readlif.reader import LifFile
from skimage.feature import blob_log
from scipy.spatial.distance import cdist

In [ ]:
# --- Blob Detection Function ---
def detect_blobs(image, min_sigma=2, max_sigma=6, threshold=0.02):
    return blob_log(image, min_sigma=min_sigma, max_sigma=max_sigma, num_sigma=10, threshold=threshold)

In [ ]:
# --- Match Ki67 blobs to DAPI blobs ---
def match_blobs(dapi_blobs, ki67_blobs, max_distance=10):
    matched_ki67 = []
    dapi_coords = np.array([[y, x] for y, x, _ in dapi_blobs])
    ki67_coords = np.array([[y, x] for y, x, _ in ki67_blobs])

    if len(ki67_coords) == 0 or len(dapi_coords) == 0:
        return [], dapi_blobs  # no matches possible

    # distances = cdist(ki67_coords, dapi_coords)
    # matched_indices = np.argmin(distances, axis=1)
    # ki67_positive = set()
    
    # for i, dapi_idx in enumerate(matched_indices):
    #     if distances[i, dapi_idx] <= max_distance:
    #         ki67_positive.add(dapi_idx)

    ki67_positive = set()
    
    for idx, blob in enumerate(ki67_blobs):
        ki67_positive.add(idx)
            
    ki67_negative = [blob for idx, blob in enumerate(dapi_blobs) if idx not in ki67_positive]
    return list(ki67_positive), ki67_negative

In [ ]:
# --- Paths ---
src_path = './All'
output_dir = './All'
os.makedirs(output_dir, exist_ok=True)

# --- Storage ---
all_records = []

In [ ]:
# --- File Loop ---
for root, _, files in os.walk(src_path):
    for filename in files:
        if filename.endswith(".lif"):
            lif_file = LifFile(os.path.join(root, filename))
            for image in lif_file.get_iter_image():
                if len(image.dims_n) < 3 or image.channels < 2:
                    continue

                print(f"🧪 Processing: {image.name}")
                z_slices = image.dims_n[3]
                dapi_stack = np.array([image.get_plane(c=0, requested_dims={3: i}) for i in range(z_slices)])
                ki67_stack = np.array([image.get_plane(c=1, requested_dims={3: i}) for i in range(z_slices)])
                
                dapi_proj = np.max(dapi_stack, axis=0)
                ki67_proj = np.max(ki67_stack, axis=0)

                # Normalize for blob detection
                dapi_norm = (dapi_proj - np.min(dapi_proj)) / (np.ptp(dapi_proj) + 1e-8)
                ki67_norm = (ki67_proj - np.min(ki67_proj)) / (np.ptp(ki67_proj) + 1e-8)
                
                # Detect blobs
                dapi_blobs = detect_blobs(dapi_norm, min_sigma=2, max_sigma=6, threshold=0.02)
                ki67_blobs = detect_blobs(ki67_norm, min_sigma=2, max_sigma=6, threshold=0.04)

                ki67_pos_indices, ki67_neg_blobs = match_blobs(dapi_blobs, ki67_blobs)

                total_dapi_cells = len(dapi_blobs)
                total_ki67_pos = len(ki67_pos_indices)
                total_ki67_neg = len(ki67_neg_blobs)

                ki67_percent = (total_ki67_pos / (total_ki67_pos + total_ki67_neg) * 100) if (total_ki67_pos + total_ki67_neg) > 0 else 0

                # Visualization (optional)
                overlay = np.stack([dapi_norm, ki67_norm, np.zeros_like(dapi_norm)], axis=-1)

                for idx, blob in enumerate(ki67_blobs):
                    y, x, r = blob
                    cv2.circle(overlay, (int(x), int(y)), int(r)+2, (0, 1, 0), 1)  # green for Ki67+
                        
                for idx, blob in enumerate(dapi_blobs):
                    y, x, r = blob
                    if idx in ki67_pos_indices:
                        continue
                    else:
                        cv2.circle(overlay, (int(x), int(y)), int(r)+2, (1, 1, 1), 1)  # white for Ki67-

                preview_path = os.path.join(output_dir, f"{image.name}_annotated.png")
                plt.imsave(preview_path, np.clip(overlay, 0, 1))

                # Optional display preview
                plt.figure(figsize=(6, 6))
                plt.imshow(overlay)
                plt.title(f"{image.name} - Total: {total_dapi_cells}, Ki67+: {total_ki67_pos}")
                plt.axis('off')
                plt.show()

               
                # Store results
                all_records.append({
                    "Image ID": image.name,
                    "DAPI cell count": total_dapi_cells,
                    "Ki67+ cell count": total_ki67_pos,
                    "Ki67- cell count": total_ki67_neg,
                    "Ki67+ %": round(ki67_percent, 2)
                })

# Save Excel
records_df = pd.DataFrame(all_records)
records_df.to_excel(os.path.join(output_dir, "Ki67_analysis_final.xlsx"), index=False)
print("✅ Analysis complete and saved.")